# Getting Started with spectraflex

**spectraflex** identifies transfer functions from OrcaFlex white noise simulations and uses them for fast spectral response prediction.

## The idea

In traditional offshore analysis, you run a separate time-domain simulation for every sea state. This is expensive — each 3-hour storm requires a full nonlinear time-domain solve.

spectraflex takes a different approach:

1. **Run once** — Run a single white noise simulation in OrcaFlex (all frequencies excited simultaneously)
2. **Identify** — Extract the transfer function H(f) using cross-spectral analysis
3. **Predict many** — Multiply H(f) by any wave spectrum to instantly get response statistics

The transfer function captures how the system amplifies and phase-shifts each frequency component. Once identified, predicting the response to a new sea state is just a spectrum multiplication — milliseconds instead of hours.

## Installation

```bash
# Clone the repository
git clone <repo-url>
cd spectraflex

# Install in development mode
pip install -e ".[dev]"
```

You will also need:
- **OrcFxAPI** and an OrcaFlex licence for running simulations and extracting results
- **matplotlib** for plotting (included in dev dependencies)

## Step 1: Generate a white noise model

The first step is to create an OrcaFlex model that uses a **white noise wave** — a wave with equal energy at all frequencies within a specified band. This excites the system across all relevant frequencies simultaneously, so a single simulation reveals the full frequency response.

spectraflex generates YAML variation files that modify your existing base model. You provide:
- A **base model** (your normal `.dat` or `.yml` file with the structure already set up)
- **Hs** — significant wave height (sets the energy level)
- **freq_range** — the frequency band `(f_min, f_max)` in Hz
- **duration** — simulation length in seconds (longer = better frequency resolution)

In [ ]:
from spectraflex.orcaflex import white_noise

# Generate a YAML variation file from your base model
# Replace "path/to/your/base_model.dat" with the actual path to your OrcaFlex model
output_path = white_noise.generate(
    template="path/to/your/base_model.dat",
    hs=2.0,                       # Significant wave height [m] — sets energy level
    freq_range=(0.02, 0.25),      # Frequency band [Hz] (periods ~4s to ~50s)
    duration=4096.0,              # Simulation duration [s] — longer = better resolution
    wave_direction=0.0,           # Wave direction [deg]
    output_dir="./white_noise",   # Where to save the generated file
)

print(f"Generated: {output_path}")

This creates a YAML file that sets the wave type to "Response calculation" (OrcaFlex's white noise wave type) and configures the frequency range and duration. You can open it in a text editor to see exactly what it changes.

**Choosing parameters:**
- **freq_range**: Should cover all wave frequencies relevant to your system. `(0.02, 0.25)` Hz covers periods from 4s to 50s, which is sufficient for most offshore applications.
- **duration**: Longer simulations give better frequency resolution (`df = 1/T`). 4096s is a good starting point. For very low frequency responses, consider longer durations.
- **Hs**: The energy level of the white noise. This matters for nonlinear systems (e.g. drag-dominated responses) since the transfer function depends on amplitude. Use an Hs representative of the sea states you want to predict.

## Step 2: Run the simulation

Run the generated model in OrcaFlex (GUI or batch). The YAML variation file references your base model, so OrcaFlex loads the base and applies the white noise wave settings on top.

Once the simulation completes, you'll have a `.sim` file containing the full time history results.

## Step 3: Identify the transfer function

This is where spectraflex does its main work. Given a completed `.sim` file, `identify.from_sim()` extracts the wave elevation and response time histories, then computes the transfer function using cross-spectral analysis (Welch's method).

You specify which results to extract as a list of dictionaries:

In [ ]:
from spectraflex import identify

# Define which results to extract from the simulation
results = [
    {
        "object": "Riser",               # OrcaFlex object name
        "variable": "Effective Tension",  # OrcaFlex result variable
        "arclength": 0.0,                 # Arc length along the line [m]
        "label": "Tension_top",           # Friendly name for the variable
    },
    {
        "object": "Riser",
        "variable": "Rotation 1",         # Bending angle
        "end_a": True,                    # Use End A instead of arc length
        "label": "Rotation_endA",
    },
    {
        "object": "Vessel",
        "variable": "X",                  # Surge displacement
        "label": "Vessel_surge",
    },
]

# Identify transfer functions from the .sim file
# Replace with the path to your completed simulation
tf = identify.from_sim(
    sim_path="path/to/your/simulation.sim",
    results=results,
    nperseg=1024,                # FFT segment length (controls resolution vs smoothing)
    freq_range=(0.02, 0.25),    # Match the white noise frequency range
)

print(tf)

The returned `tf` is an `xarray.Dataset` containing:
- **magnitude** — `|H(f)|`, the gain at each frequency (shape: `n_freq x n_variables`)
- **phase** — `arg(H(f))`, the phase shift in radians
- **coherence** — `gamma^2(f)`, how well a linear model explains the response (0 to 1)

**About `nperseg`:** This controls the trade-off between frequency resolution and statistical smoothing. Larger values give finer frequency resolution but noisier estimates. A good starting point is 1024 — increase if you need to resolve closely-spaced peaks, decrease if the estimate looks noisy.

**About `freq_range`:** Should match the frequency range you used in the white noise generation. Outside this range the wave has no energy, so the transfer function estimate is meaningless.

## Step 4: Inspect the transfer function

Always inspect the identified transfer function before using it for predictions. The three key plots are magnitude, phase, and coherence.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_transfer_function(tf, variable_index=0):
    """Plot magnitude, phase, and coherence for a single variable."""
    f = tf.coords["frequency"].values
    var_name = str(tf.coords["variable"].values[variable_index])
    mag = tf["magnitude"].values[:, variable_index]
    phase = tf["phase"].values[:, variable_index]
    coh = tf["coherence"].values[:, variable_index]

    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

    # Magnitude
    axes[0].semilogy(f, mag, "b-", linewidth=1.5)
    axes[0].set_ylabel("|H(f)|")
    axes[0].set_title(f"Transfer Function: {var_name}")
    axes[0].grid(True, alpha=0.3)

    # Phase
    axes[1].plot(f, np.degrees(phase), "b-", linewidth=1.5)
    axes[1].set_ylabel("Phase [deg]")
    axes[1].grid(True, alpha=0.3)

    # Coherence
    axes[2].plot(f, coh, "g-", linewidth=1.5)
    axes[2].axhline(0.8, color="k", linestyle="--", alpha=0.5, label="Threshold (0.8)")
    axes[2].set_ylabel("Coherence")
    axes[2].set_xlabel("Frequency [Hz]")
    axes[2].set_ylim(0, 1.05)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    return fig


# Plot each variable (uncomment when you have a real tf)
# for i in range(len(tf.coords["variable"])):
#     plot_transfer_function(tf, variable_index=i)
#     plt.show()

**What to look for:**

- **Magnitude**: Should show physically sensible peaks (resonances) and decay at high frequencies. Erratic spikes may indicate noise or insufficient simulation duration.
- **Phase**: Should vary smoothly. Abrupt jumps of 180 degrees near resonance peaks are expected.
- **Coherence**: Values close to 1.0 mean the linear model is a good fit at that frequency. Low coherence (< 0.5) indicates nonlinear behavior, noise, or frequencies outside the white noise band. Frequencies with low coherence should be treated with caution.

You can mask out low-coherence regions before using the transfer function for predictions:

In [ ]:
# Optional: zero out the transfer function where coherence is low
# tf_clean = identify.apply_coherence_mask(tf, threshold=0.5)

## Step 5: Predict response for different sea states

This is where spectraflex pays off. With the transfer function identified, you can predict response statistics for any sea state instantly.

The prediction is based on the spectral relationship:

$$S_{yy}(f) = |H(f)|^2 \cdot S_{xx}(f)$$

where $S_{xx}$ is the wave spectrum (e.g. JONSWAP) and $S_{yy}$ is the response spectrum. From $S_{yy}$ we compute spectral moments and derive statistics like significant response height (Hs), zero-crossing period (Tz), and most probable maximum (MPM).

In [ ]:
from spectraflex import predict, spectrum

# Define a frequency array for the wave spectra
f = spectrum.frequency_array(f_min=0.02, f_max=0.3, n_freq=256)

# Define sea states to analyze
sea_states = [
    {"hs": 1.5, "tp": 7.0, "gamma": 3.3, "label": "Operational"},
    {"hs": 3.0, "tp": 9.0, "gamma": 3.3, "label": "Design"},
    {"hs": 5.0, "tp": 11.0, "gamma": 2.5, "label": "Extreme"},
]

# Predict response statistics for each sea state
duration = 3 * 3600.0  # 3-hour storm [s]

for sea in sea_states:
    # Create JONSWAP wave spectrum
    wave = spectrum.jonswap(hs=sea["hs"], tp=sea["tp"], f=f, gamma=sea["gamma"])

    # Compute response statistics for all variables at once
    stats = predict.response_statistics(tf, wave, duration=duration)

    print(f"\n--- {sea['label']} (Hs={sea['hs']}m, Tp={sea['tp']}s) ---")
    for var_name, var_stats in stats.items():
        print(f"  {var_name}:")
        print(f"    Hs = {var_stats['hs']:.3f}")
        print(f"    Tz = {var_stats['tz']:.2f} s")
        print(f"    MPM = {var_stats['mpm']:.3f}")

You can also plot the response spectrum to see which frequencies contribute most to the response:

In [ ]:
def plot_response_prediction(tf, sea_states, f, variable_index=0):
    """Plot wave spectra and predicted response spectra side by side."""
    var_name = str(tf.coords["variable"].values[variable_index])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for sea in sea_states:
        wave = spectrum.jonswap(hs=sea["hs"], tp=sea["tp"], f=f, gamma=sea["gamma"])
        resp = predict.response_spectrum(tf, wave)
        syy = resp["Syy"].values[:, variable_index]

        axes[0].plot(f, wave.values, linewidth=1.5, label=sea["label"])
        axes[1].plot(resp.coords["frequency"].values, syy, linewidth=1.5, label=sea["label"])

    axes[0].set_xlabel("Frequency [Hz]")
    axes[0].set_ylabel("S(f) [m²/Hz]")
    axes[0].set_title("Wave Spectra")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].set_xlabel("Frequency [Hz]")
    axes[1].set_ylabel("S_response(f)")
    axes[1].set_title(f"Predicted Response Spectra: {var_name}")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    return fig


# Uncomment when you have a real tf:
# plot_response_prediction(tf, sea_states, f, variable_index=0)
# plt.show()

## Self-contained example (no OrcaFlex required)

The cells above require a real OrcaFlex `.sim` file. To verify the workflow end-to-end without OrcaFlex, we can create a synthetic transfer function with known properties, generate fake time histories, identify H(f) back, and predict responses.

This is also useful as a sanity check: the identified transfer function should match the known one.

In [ ]:
from spectraflex import identify, predict, spectrum, transfer_function

# --- 1. Define a known transfer function ---
# Simple resonance at 0.08 Hz (12.5s period), typical for a floating structure
f_tf = np.linspace(0.01, 0.3, 512)
f0 = 0.08   # resonance frequency [Hz]
zeta = 0.1  # damping ratio
gain = 2.0  # static gain

omega = 2 * np.pi * f_tf
omega0 = 2 * np.pi * f0
h_true = gain / (1 - (omega / omega0) ** 2 + 2j * zeta * (omega / omega0))

print(f"True transfer function: resonance at {f0} Hz, peak |H| = {np.abs(h_true).max():.1f}")

In [ ]:
# --- 2. Generate synthetic white noise input and filtered response ---
dt = 0.5        # sample interval [s]
duration = 4096 # simulation duration [s]
n_samples = int(duration / dt)

rng = np.random.default_rng(42)
wave = rng.normal(0, 1, n_samples)

# Filter through H(f) in the frequency domain (mimics what OrcaFlex does physically)
fft_freq = np.fft.rfftfreq(n_samples, dt)
h_interp = np.interp(fft_freq, f_tf, h_true)
wave_fft = np.fft.rfft(wave)
response = np.fft.irfft(wave_fft * h_interp, n=n_samples)

print(f"Generated {n_samples} samples ({duration}s at dt={dt}s)")
print(f"Wave std: {wave.std():.3f}, Response std: {response.std():.3f}")

In [ ]:
# --- 3. Identify H(f) from the time histories ---
tf = identify.from_time_histories(
    wave_elevation=wave,
    responses={"Heave": response},
    dt=dt,
    nperseg=1024,
    freq_range=(0.01, 0.3),
)

# Compare identified vs true
tf_freq = tf.coords["frequency"].values
tf_mag = tf["magnitude"].values[:, 0]
tf_phase = tf["phase"].values[:, 0]
tf_coh = tf["coherence"].values[:, 0]

# Interpolate true H to the same frequencies
mag_true = np.interp(tf_freq, f_tf, np.abs(h_true))
phase_true = np.interp(tf_freq, f_tf, np.angle(h_true))

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].semilogy(tf_freq, mag_true, "r--", linewidth=2, label="True")
axes[0].semilogy(tf_freq, tf_mag, "b-", linewidth=1.5, alpha=0.8, label="Identified")
axes[0].set_ylabel("|H(f)|")
axes[0].set_title("Transfer Function Identification: True vs Identified")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(tf_freq, np.degrees(phase_true), "r--", linewidth=2, label="True")
axes[1].plot(tf_freq, np.degrees(tf_phase), "b-", linewidth=1.5, alpha=0.8, label="Identified")
axes[1].set_ylabel("Phase [deg]")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(tf_freq, tf_coh, "g-", linewidth=1.5)
axes[2].axhline(0.8, color="k", linestyle="--", alpha=0.5)
axes[2].set_ylabel("Coherence")
axes[2].set_xlabel("Frequency [Hz]")
axes[2].set_ylim(0, 1.05)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nMean coherence: {tf_coh.mean():.3f}")
print(f"Magnitude MAPE (where coh > 0.8): ", end="")
good = tf_coh > 0.8
if good.sum() > 0:
    mape = np.mean(np.abs(tf_mag[good] - mag_true[good]) / mag_true[good])
    print(f"{100 * mape:.1f}%")
else:
    print("N/A")

In [ ]:
# --- 4. Predict responses for different sea states ---
f_pred = spectrum.frequency_array(f_min=0.02, f_max=0.3, n_freq=256)
duration_storm = 3 * 3600.0  # 3-hour storm

sea_states = [
    {"hs": 1.5, "tp": 7.0, "gamma": 3.3, "label": "Operational"},
    {"hs": 3.0, "tp": 9.0, "gamma": 3.3, "label": "Design"},
    {"hs": 5.0, "tp": 11.0, "gamma": 2.5, "label": "Extreme"},
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

print(f"{'Sea State':<14} {'Wave Hs':>10} {'Wave Tp':>10} {'Resp Hs':>10} {'Resp MPM':>10}")
print("-" * 56)

for sea in sea_states:
    wave_spec = spectrum.jonswap(hs=sea["hs"], tp=sea["tp"], f=f_pred, gamma=sea["gamma"])
    stats = predict.response_statistics(tf, wave_spec, duration=duration_storm)
    s = stats["Heave"]

    print(f"{sea['label']:<14} {sea['hs']:>10.1f} {sea['tp']:>10.1f} {s['hs']:>10.3f} {s['mpm']:>10.3f}")

    resp = predict.response_spectrum(tf, wave_spec)
    axes[0].plot(f_pred, wave_spec.values, linewidth=1.5, label=sea["label"])
    axes[1].plot(resp.coords["frequency"].values, resp["Syy"].values[:, 0],
                 linewidth=1.5, label=sea["label"])

axes[0].set_xlabel("Frequency [Hz]")
axes[0].set_ylabel("S(f) [m²/Hz]")
axes[0].set_title("Wave Spectra (JONSWAP)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Frequency [Hz]")
axes[1].set_ylabel("S_response(f)")
axes[1].set_title("Predicted Response Spectra")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Next steps

- **Save/load transfer functions**: Use `spectraflex.io.netcdf` to save and load transfer functions as NetCDF files for reuse.
- **Batch generation**: Use `white_noise.generate_batch()` to create models for multiple wave directions and current speeds.
- **Transfer function library**: Use `spectraflex.TransferFunctionLibrary` to organize transfer functions by environmental parameters and look up the right one for a given sea state.
- **Fatigue**: Use `spectraflex.fatigue` for spectral fatigue damage calculations using DNV-RP-C203 S-N curves.
- **Time series synthesis**: Use `predict.synthesize_timeseries()` to generate synthetic response time series from the transfer function.

See the other examples in this directory for more details:
- `spectral_prediction.py` — predicting responses across multiple sea states
- `library_workflow.py` — organizing and querying transfer function libraries
- `fatigue_comparison.py` — spectral fatigue analysis with OrcaFlex validation